# 📌 **02C — Procesamiento de Mosaicos (GCS o Google Drive)**

Este notebook procesa los archivos **GeoTIFF** exportados desde GEE (paso 02B).
Une los tiles exportados en mosaicos completos por año y los guarda en el destino
configurado, listos para ser descargados y usados en R (paso 02D → 03A).

---

## ⚙️ Fuente de datos — `source_target`

Debe coincidir con el `exportTarget` usado en **02B**:

| `source_target` | Lee desde | Escribe en |
|---|---|---|
| `'gcs'` | Google Cloud Storage | Google Cloud Storage |
| `'drive'` | Google Drive | Google Drive |

---

## 🎯 Requisitos previos
- Haber corrido **02A** (native mask en GEE) y **02B** (exportación de TIFFs)
- Para `'gcs'`: bucket configurado y permisos de lectura/escritura
- Para `'drive'`: carpeta `mapbiomas-colombia-degradacion` en tu Drive

## 📜 Flujo
| Celda | Acción |
|---|---|
| 1 | Instalar dependencias y autenticar |
| 2 | Configurar fuente y destino |
| 3 | Definir funciones auxiliares |
| 4 | Cargar información de archivos disponibles |
| 5 | Seleccionar bandas (interfaz interactiva) |
| 6 | Ejecutar mosaico y guardar resultados |

In [ ]:
# Celda 1 — Instalar dependencias y autenticar
!apt-get update -q
!apt-get install -y -q gdal-bin python3-gdal
!pip install gdal tqdm google-cloud-storage ipywidgets -q

from google.colab import auth, drive
import os, subprocess, time, re, shutil
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output

auth.authenticate_user()
print('✅ Autenticación Google lista.')

In [ ]:
# Celda 2 — Configuración de fuente y destino
# ─────────────────────────────────────────────────────────────
# Cambiar source_target para que coincida con exportTarget de 02B
#   'drive' → leer/escribir en Google Drive
#   'gcs'   → leer/escribir en Google Cloud Storage
# ─────────────────────────────────────────────────────────────
source_target = 'drive'

# --- Google Drive ---
drive_folder_name = 'mapbiomas-colombia-degradacion'   # misma carpeta que en 02B
drive_mount_path  = '/content/drive'
drive_input_sub   = 'temp'          # subcarpeta con los tiles exportados por 02B
drive_output_sub  = 'mosaicos'      # subcarpeta donde se guardarán los mosaicos

# --- Google Cloud Storage ---
bucket_name   = 'mbcolombia-degradacion'               # bucket GCS proyecto cloud-ee-lmedinaj
input_folder  = 'AUXILIARES/DEGRADACION/COL_3/temp'
output_folder = 'AUXILIARES/DEGRADACION/COL_3/'

# ─── Montar Drive si es necesario ────────────────────────────
if source_target == 'drive':
    drive.mount(drive_mount_path)
    drive_base        = os.path.join(drive_mount_path, 'MyDrive', drive_folder_name)
    drive_input_path  = os.path.join(drive_base, drive_input_sub)
    drive_output_path = os.path.join(drive_base, drive_output_sub)
    os.makedirs(drive_output_path, exist_ok=True)
    print(f'Drive montado.')
    print(f'  Entrada:  {drive_input_path}')
    print(f'  Salida:   {drive_output_path}')
else:
    from google.cloud import storage
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    print(f'GCS configurado.')
    print(f'  Bucket:  {bucket_name}')
    print(f'  Entrada: {input_folder}')
    print(f'  Salida:  {output_folder}')

In [ ]:
# Celda 3 — Funciones auxiliares

# ── Utilidades compartidas ────────────────────────────────────

def format_time(seconds):
    days=int(seconds//86400); hours=int((seconds%86400)//3600)
    minutes=int((seconds%3600)//60); secs=int(seconds%60)
    if days>0:    return f'{days}d {hours}h {minutes}m {secs}s'
    if hours>0:   return f'{hours}h {minutes}m {secs}s'
    if minutes>0: return f'{minutes}m {secs}s'
    return f'{secs}s'

def mosaic_images(input_files, output_file):
    """Crea un mosaico GeoTIFF con GDAL a partir de una lista de tiles."""
    if not input_files:
        print('[ERROR] Sin archivos de entrada.'); return False
    with open('input_files.txt', 'w') as f:
        f.writelines(fp + '\n' for fp in input_files)
    vrt = 'temp.vrt'
    if os.system(f'gdalbuildvrt -input_file_list input_files.txt {vrt}') != 0:
        print('[ERROR] gdalbuildvrt falló.'); return False
    result = subprocess.run([
        'gdal_translate', '-of', 'GTiff', '-ot', 'Byte',
        '-co', 'TILED=YES', '-co', 'COMPRESS=LZW', '-co', 'PREDICTOR=2',
        '-co', 'ZLEVEL=9',  '-co', 'NUM_THREADS=ALL_CPUS',
        '-co', 'BIGTIFF=YES', '-co', 'SPARSE_OK=TRUE',
        vrt, output_file
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for tmp in [vrt, 'input_files.txt']:
        if os.path.exists(tmp): os.remove(tmp)
    if result.returncode != 0:
        print('[ERROR] gdal_translate falló:\n', result.stderr); return False
    return os.path.exists(output_file)

def parse_file_info(filename):
    """Extrae data_type, band e is_mosaic del nombre de archivo."""
    base      = filename[:-4] if filename.endswith('.tif') else filename
    is_mosaic = bool(re.search(r'\d{10}-\d{10}$', base))
    clean     = base[:-21] if is_mosaic else base
    parts     = clean.split('-')
    if len(parts) < 2: return None, None, None
    return parts[0], parts[1], is_mosaic


# ── Funciones DRIVE ───────────────────────────────────────────

def list_files_drive(folder_path):
    """Lista archivos .tif en una carpeta de Drive (ruta local montada)."""
    files = []
    if not os.path.exists(folder_path):
        print(f'[AVISO] Carpeta no encontrada: {folder_path}'); return files
    for fname in os.listdir(folder_path):
        if fname.lower().endswith('.tif'):
            files.append(os.path.join(folder_path, fname))
    return files

def mosaic_exists_drive(output_path, data_type, band):
    out = os.path.join(output_path, data_type, f'{data_type}-{band}.tif')
    return os.path.exists(out)

def load_data_info_drive(input_path, output_path):
    files     = list_files_drive(input_path)
    data_info = {}
    print(f'[INFO] Archivos encontrados en Drive: {len(files)}')
    for f in files:
        fname                   = os.path.basename(f)
        data_type, band, is_m   = parse_file_info(fname)
        if data_type is None: continue
        mosaic_exists = mosaic_exists_drive(output_path, data_type, band)
        if data_type not in data_info:
            data_info[data_type] = {'bands': {}, 'requires_mosaic': is_m}
        if band not in data_info[data_type]['bands']:
            data_info[data_type]['bands'][band] = {'files': [], 'mosaic_exists': mosaic_exists}
        if is_m: data_info[data_type]['bands'][band]['files'].append(f)
        else:    data_info[data_type]['bands'][band]['file'] = f
    _print_summary(data_info)
    return data_info

def process_mosaics_drive(selected_bands, data_info, output_path):
    start = time.time()
    total = sum(len(b) for b in selected_bands.values()); done = 0
    print(f'\n{"="*60}\n[🚀 INICIO] {total} bandas seleccionadas.\n{"="*60}')
    for data_type, bands in selected_bands.items():
        for band in bands:
            print(f'\n{"-"*60}\n[🎵] {data_type}-{band}  ({done}/{total})')
            out_dir  = os.path.join(output_path, data_type)
            os.makedirs(out_dir, exist_ok=True)
            out_file = os.path.join(out_dir, f'{data_type}-{band}.tif')
            if mosaic_exists_drive(output_path, data_type, band):
                print(f'[✅ YA EXISTE] Omitiendo.'); done += 1; continue
            is_m = data_info[data_type]['requires_mosaic']
            if is_m:
                tiles = data_info[data_type]['bands'][band].get('files', [])
                if not tiles: print('[ERROR] Sin tiles.'); continue
                tmp = f'/content/temp/{data_type}-{band}.tif'
                os.makedirs('/content/temp', exist_ok=True)
                if not mosaic_images(tiles, tmp):
                    print('[ERROR] Mosaico falló.'); continue
                shutil.move(tmp, out_file)
                print(f'[✅ GUARDADO] {out_file}')
            else:
                src = data_info[data_type]['bands'][band]['file']
                shutil.copy2(src, out_file)
                print(f'[✅ COPIADO]  {out_file}')
            done += 1
            elapsed = time.time() - start
            print(f'[⏳] {format_time(elapsed)} transcurrido  |  '
                  f'~{format_time(elapsed/done*(total-done))} restante')
    print(f'\n{"="*60}\n[✅ LISTO] {done}/{total} bandas.  '
          f'Tiempo total: {format_time(time.time()-start)}\n{"="*60}')


# ── Funciones GCS ─────────────────────────────────────────────

def list_files_gcs(bucket_name, folder_path):
    files = []
    for blob in client.list_blobs(bucket_name, prefix=folder_path):
        if blob.name.lower().endswith('.tif'): files.append(blob.name)
    return files

def mosaic_exists_gcs(bucket_name, output_folder, data_type, band):
    path = f"{output_folder.rstrip('/')}/{data_type}/{data_type}-{band}.tif"
    return bucket.blob(path).exists()

def upload_file_gcs(source, destination):
    blob = bucket.blob(destination)
    if source.startswith('/content/'):
        blob.upload_from_filename(source)
        print(f'[UPLOAD ✅] → {destination}')
    else:
        bucket.blob(source).copy_to(bucket.blob(destination))
        print(f'[COPIA ✅]  → {destination}')

def load_data_info_gcs(bucket_name, input_folder, output_folder):
    files     = list_files_gcs(bucket_name, input_folder)
    data_info = {}
    print(f'[INFO] Archivos en GCS: {len(files)}')
    for f in files:
        fname                 = os.path.basename(f)
        data_type, band, is_m = parse_file_info(fname)
        if data_type is None: continue
        try:    mosaic_exists = mosaic_exists_gcs(bucket_name, output_folder, data_type, band)
        except: mosaic_exists = False
        if data_type not in data_info:
            data_info[data_type] = {'bands': {}, 'requires_mosaic': is_m}
        if band not in data_info[data_type]['bands']:
            data_info[data_type]['bands'][band] = {'files': [], 'mosaic_exists': mosaic_exists}
        if is_m: data_info[data_type]['bands'][band]['files'].append(f)
        else:    data_info[data_type]['bands'][band]['file'] = f
    _print_summary(data_info)
    return data_info

def process_mosaics_gcs(selected_bands, data_info, bucket_name, input_folder, output_folder):
    start = time.time()
    total = sum(len(b) for b in selected_bands.values()); done = 0
    print(f'\n{"="*60}\n[🚀 INICIO] {total} bandas seleccionadas.\n{"="*60}')
    for data_type, bands in selected_bands.items():
        for band in bands:
            print(f'\n{"-"*60}\n[🎵] {data_type}-{band}  ({done}/{total})')
            if mosaic_exists_gcs(bucket_name, output_folder, data_type, band):
                print('[✅ YA EXISTE] Omitiendo.'); done += 1; continue
            out_name   = f'{data_type}/{data_type}-{band}.tif'
            out_remote = f"{output_folder.rstrip('/')}/{out_name}"
            is_m = data_info[data_type]['requires_mosaic']
            if is_m:
                tiles    = [f'/vsigs/{bucket_name}/{fp}' for fp in data_info[data_type]['bands'][band].get('files', [])]
                out_local = f'/content/temp/{out_name}'
                os.makedirs(os.path.dirname(out_local), exist_ok=True)
                if not mosaic_images(tiles, out_local):
                    print('[ERROR] Mosaico falló.'); continue
                upload_file_gcs(out_local, out_remote)
                try: os.remove(out_local)
                except: pass
            else:
                src  = f"/vsigs/{bucket_name}/{data_info[data_type]['bands'][band]['file']}"
                upload_file_gcs(src, out_remote)
            done += 1
            elapsed = time.time() - start
            print(f'[⏳] {format_time(elapsed)} transcurrido  |  '
                  f'~{format_time(elapsed/done*(total-done))} restante')
    print(f'\n{"="*60}\n[✅ LISTO] {done}/{total} bandas.  '
          f'Tiempo total: {format_time(time.time()-start)}\n{"="*60}')


# ── Enrutadores ───────────────────────────────────────────────

def _print_summary(data_info):
    print('\n🔍 Tipos encontrados:', list(data_info.keys()))
    for dt, info in data_info.items():
        print(f'  🗂️  {dt} — bandas: {list(info["bands"].keys())}')

def load_data_info():
    if source_target == 'drive':
        return load_data_info_drive(drive_input_path, drive_output_path)
    else:
        return load_data_info_gcs(bucket_name, input_folder, output_folder)

def process_mosaics(selected_bands, data_info):
    if source_target == 'drive':
        process_mosaics_drive(selected_bands, data_info, drive_output_path)
    else:
        process_mosaics_gcs(selected_bands, data_info, bucket_name, input_folder, output_folder)

print('✅ Funciones cargadas.')

In [ ]:
# Celda 4 — Cargar información de archivos disponibles
data_info = load_data_info()

In [ ]:
# Celda 5 — Interfaz interactiva para seleccionar bandas

files_count  = sum(len(b['files']) for d in data_info.values() for b in d['bands'].values())
mosaic_count = sum(1 for d in data_info.values() for b in d['bands'].values() if b['mosaic_exists'])
bands_per_dt = {dt: len(info['bands']) for dt, info in data_info.items()}
src_label    = f'Drive → {drive_input_path}' if source_target == 'drive' else f'GCS → {bucket_name}/{input_folder}'

met_label = widgets.HTML(f"""
<p><b>Fuente:</b> {src_label}<br>
<b>Total de archivos:</b> {files_count}<br>
<b>Bandas por tipo:</b> {', '.join([f"{dt}: {c}" for dt,c in bands_per_dt.items()])}<br>
<b>Mosaicos ya generados:</b> {mosaic_count}</p>
""")

data_type_checkboxes = {}
accordion_items, accordion_titles = [], []

def toggle_dt(dt):
    vals = [cb.value for cb in data_type_checkboxes[dt].values() if not cb.disabled]
    nv   = False if (vals and all(vals)) else True
    for cb in data_type_checkboxes[dt].values():
        if not cb.disabled: cb.value = nv

for dt, info in data_info.items():
    data_type_checkboxes[dt] = {}
    rows = []
    for band, bi in info['bands'].items():
        cb = widgets.Checkbox(
            value=not bi['mosaic_exists'],
            description=f"{band}{' ⚠️' if bi['mosaic_exists'] else ''}",
            disabled=bi['mosaic_exists']
        )
        data_type_checkboxes[dt][band] = cb
        rows.append(cb)
    btn = widgets.Button(description='Activar/Desactivar Tipo', button_style='info',
                         layout=widgets.Layout(width='auto'))
    def make_cb(d): return lambda b: toggle_dt(d)
    btn.on_click(make_cb(dt))
    accordion_items.append(widgets.VBox([btn] + rows))
    accordion_titles.append(dt)

accordion = widgets.Accordion(children=accordion_items)
for i, t in enumerate(accordion_titles): accordion.set_title(i, t)

toggle_all = widgets.Button(description='Activar/Desactivar Todo', button_style='info')
def on_toggle_all(b):
    all_v = [cb.value for dt in data_type_checkboxes for cb in data_type_checkboxes[dt].values() if not cb.disabled]
    nv    = False if (all_v and all(all_v)) else True
    for dt in data_type_checkboxes:
        for cb in data_type_checkboxes[dt].values():
            if not cb.disabled: cb.value = nv
toggle_all.on_click(on_toggle_all)

display(met_label,
        widgets.HTML('<p>Bandas marcadas con ⚠️ ya fueron procesadas y no se repetirán.</p>'),
        toggle_all, accordion)

In [ ]:
# Celda 6 — Ejecutar mosaico con las bandas seleccionadas

selected_bands = {}
for dt, bands_dict in data_type_checkboxes.items():
    sel = [band for band, cb in bands_dict.items() if cb.value and not cb.disabled]
    if sel: selected_bands[dt] = sel

if not selected_bands:
    print('Ninguna banda seleccionada.')
else:
    print('Bandas seleccionadas:', selected_bands)
    process_mosaics(selected_bands, data_info)